# Modelování latentních faktorů úvěrového rizika pomocí PROC HPPLS

## Shrnutí

Retailová banka chce predikovat tři korelované ukazatele úvěrového rizika — čerpání kreditní karty, vývoj poměru dluhu k příjmu a proxy pravděpodobnosti selhání — z rozsáhlé sady vysoce kolineárních prediktorů typu bureau a makroekonomických prediktorů. Obyčejná regrese při této kolinearitě selhává, proto tento notebook používá **PROC HPPLS** (vysoce výkonné částečné nejmenší čtverce) k extrakci několika latentních faktorů, které společně vysvětlují prediktory i všechny tři odezvy, a poté ověřuje počet faktorů pomocí van der Voetova křížového ověřovacího testu na vyčleněném segmentu portfolia.

## Zdroje dat

Všechna data jsou generována synteticky přímo v notebooku pomocí `call streaminit(20240531)` — bez externích souborů nebo síťového přístupu.

| Datová sada | Řádky | Proměnná | Role | Popis |
|---------|------|----------|------|-------------|
| `credit` | 600 | `CustomerID` | ID | Unikátní klíč zákazníka přenesený do skórovaného výstupu |
| | | `Segment` | CLASS prediktor | Segment portfolia: Drobní, Movití, Firemní |
| | | `b1`–`b12` | Prediktory | 12 kolineárních měsíčních behaviorálních signálů typu bureau |
| | | `RatePctChg` | Prediktor | Expozice zákazníka vůči změně úrokové sazby |
| | | `InquiryCount` | Prediktor | Počet nedávných tvrdých úvěrových dotazů |
| | | `Utilization` | Odezva | Čerpání revolvingového úvěru (%) |
| | | `DTIChange` | Odezva | Změna poměru dluhu k příjmu |
| | | `DefaultProp` | Odezva | Proxy pravděpodobnosti selhání (0–1) |
| | | `Role` | Rozdělení | Validační příznak TRÉNINK (~75 %) / TEST (~25 %) |

# Modelování latentních faktorů úvěrového rizika pomocí PROC HPPLS

Banky se běžně potýkají s problémem **širokého, kolineárního souboru prediktorů**: desítky měsíčních atributů bureau, makroekonomických expozic a behaviorálních signálů, které se pohybují společně a slouží k predikci několika rizikových ukazatelů, jež jsou samy o sobě korelované. Obyčejné nejmenší čtverce jsou zde nestabilní, protože matice prediktorů je téměř singulární.

**Částečné nejmenší čtverce (PLS)** tento problém řeší extrakcí malého počtu latentních faktorů z *křížové kovariance* prediktorů (X) a odezev (Y), takže faktory jsou vyladěny k predikci výsledků — nejen k souhrnu X. `PROC HPPLS` je vysoce výkonný protějšek `PROC PLS`, který přidává vícevláknové provádění, rozdělení dat pro validaci a van der Voetův randomizační test pro volbu počtu faktorů.

V tomto notebooku sestavíme jediný PLS model, který současně predikuje tři korelované odezvy úvěrového rizika, a pomocí vyčleněného validačního segmentu ověříme, kolik latentních faktorů data skutečně podporují.

## Krok 1 — Generování syntetického panelu úvěrového rizika

Simulujeme 600 zákazníků. Dva neměřené latentní faktory (`stress` a `tenure`) generují dvanáct kolineárních signálů bureau `b1`–`b12` plus expozice vůči sazbě a dotazům — přesně strukturu vysoké korelace, pro kterou je PLS navrženo. Tři odezvy (`Utilization`, `DTIChange`, `DefaultProp`) jsou různé lineární kombinace stejných faktorů, takže jsou také korelované. Příznak `Role` vyčleňuje zhruba čtvrtinu portfolia jako validační segment.

In [1]:
data credit;
   CALL streaminit(20240531);
   DÉLKA Segment $16 Role $10;
   ŠTÍTEK CustomerID = "ID zákazníka"
         Role = "Skupina";
   OPAKUJ CustomerID = 1 TO 600;
      /* segment zákazníka (kategoriální prediktor) */
      u = rand('uniform');
      KDYŽ u < 0.34 PAK Segment = 'Drobní';
      JINAK KDYŽ u < 0.70 PAK Segment = 'Movití';
      JINAK Segment = 'Firemní';

      /* neměřené makro / behaviorální faktory (kolineární) */
      stress = rand('normal', 0, 1);
      tenure = rand('normal', 0, 1);

      /* 12 kolineárních měsíčních prediktorů bureau b1-b12 */
      POLE b{12} b1-b12;
      OPAKUJ j = 1 TO 12;
         b{j} = 0.9*stress + 0.6*tenure
                + 0.4*rand('normal', 0, 1) + 0.02*j;
      KONEC;

      /* makro kovariáty, také vázané na faktory */
      RatePctChg   = 1.5*stress + rand('normal', 0, 0.5);
      InquiryCount = round( max(0, 4 + 2.5*stress
                               + rand('normal', 0, 1)) );

      /* tři korelované odezvy úvěrového rizika */
      Utilization = 45 + 12*stress - 6*tenure
                    + rand('normal', 0, 4);
      DTIChange   = 2.5*stress - 1.5*tenure
                    + rand('normal', 0, 1);
      DefaultProp = 0.08 + 0.05*stress - 0.03*tenure
                    + rand('normal', 0, 0.02);
      KDYŽ DefaultProp < 0 PAK DefaultProp = 0;

      /* vyčlenění ~25 % jako validační segment */
      KDYŽ rand('uniform') < 0.25 PAK Role = 'TEST';
      JINAK Role = 'TRÉNINK';

      VÝSTUP;
   KONEC;
   ODSTRANIT u stress tenure j;
SPUSTIT;


NOTE: DATA credit

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote credit (100 rows, 20 columns).
NOTE: DATA elapsed:
  wall  0.36 seconds
  cpu   0.36 seconds


## Krok 2 — Proložení vícerozměrného PLS modelu s křížovou validací

Hlavní volání demonstruje klíčové příkazy `PROC HPPLS` a volby, které by risk modelář skutečně použil:

- **`MODEL`** uvádí na levé straně všechny tři odezvy a na pravé straně celou kolineární sadu prediktorů; **`/ solution`** vypíše konečné predikční koeficienty v původní škále.
- **`CLASS Segment`** vnáší segment portfolia jako kategoriální prediktor (musí předcházet `MODEL`).
- **`ID CustomerID`** přenáší klíč zákazníka do skórovaného výstupu.
- **`CVTEST(stat=press ...)`** spouští van der Voetův randomizační test pro objektivní volbu počtu faktorů namísto odhadu pohledem; `seed=` zajišťuje reprodukovatelnost.
- **`PARTITION rolevar=Role(...)`** proloží a skóruje na trénovacích řádcích a vyčlení segment `TEST`, takže křížově validovaný PRESS je měřen mimo trénovací vzorek.
- **`VARSS`** a **`DETAILS`** vykazují, kolik variability X a Y vysvětluje každý další faktor.
- **`OUTPUT`** zapisuje predikované hodnoty, rezidua, X-skóre a PRESS pro proložené (trénovací) řádky do skórované datové sady, klíčované podle `CustomerID`.

In [2]:
PROCEDURA hppls data=credit nfac=8
           cvtest(stat=press pval=0.10 nsamp=500 seed=4242)
           varss details;
   TŘÍDA Segment;
   id CustomerID;
   MODEL Utilization DTIChange DefaultProp =
         b1-b12 RatePctChg InquiryCount Segment / SOLUTION;
   partition rolevar=Role(train='TRÉNINK' test='TEST');
   VÝSTUP out=scored predicted=Pred residual=Resid
          xscore=Factor press=Press role=AssignedRole;
SPUSTIT;


The HPPLS Procedure

Method: PLS
Algorithm: NIPALS
Number of Observations Read: 100
Number of Observations Used: 76
Number of Training Observations: 76
Number of Test Observations:     24

Class Level Information

  Class Segment: 3 levels: Drobní Firemní Movití

Response Variable(s): Utilization DTIChange DefaultProp
Predictor Variable(s): b1 b2 b3 b4 b5 b6 b7 b8 b9 b10 b11 b12 RatePctChg InquiryCount Segment

Number of Extracted Factors: 8

Percent Variation Accounted for by HPPLS Factors

            Model Effects      Dependent Variables
 Factor   Current  Total       Current  Total
    1     74.1567 74.1567     25.5160 25.5160
    2      5.2435 79.4002     45.2909 70.8069
    3      6.6707 86.0709      2.8871 73.6940
    4      4.9359 91.0068      1.5828 75.2768
    5      1.2368 92.2436      1.2734 76.5502
    6      1.1201 93.3638      0.9318 77.4820
    7      0.9793 94.3431      0.3379 77.8199
    8      0.6559 94.9991      0.1380 77.9578

Variation Accounted for by Variable



NOTE: PROC HPPLS data=credit

NOTE: PROC HPPLS completed.


## Krok 3 — Souhrn predikovaného rizikového profilu

S proloženým modelem profilujeme predikované výsledky napříč portfoliem. `PROC MEANS` vykazuje střední hodnotu a rozptyl každé predikované odezvy, aby si tým řízení rizik mohl ověřit škálu (např. predikované čerpání se středem kolem 40 %, proxy selhání blízko předpokládané základní míry 8 %).

In [3]:
PROCEDURA PRŮMĚRY data=scored mean std MIN MAX maxdec=3;
   ŠTÍTEK Pred_Utilization = "Predikce čerpání úvěru"
         Pred_DTIChange = "Predikce změny DTI"
         Pred_DefaultProp = "Predikce pravděpodobnosti selhání";
   PROMĚNNÁ Pred_Utilization Pred_DTIChange Pred_DefaultProp;
SPUSTIT;

                                                  The MEANS Procedure

 Variable          Label                                          Mean     Std Dev     Minimum     Maximum
 ---------------------------------------------------------------------------------------------------------
 PRED_UTILIZATION  Predikce čerpání úvěru                       47.405      10.897      29.053      82.670
 PRED_DTICHANGE    Predikce změny DTI                            0.649       2.502      -3.863       9.184
 PRED_DEFAULTPROP  Predikce pravděpodobnosti selhání             0.090       0.049       0.008       0.234
 ---------------------------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Krok 4 — Kontrola jednotlivých skórovaných zákazníků

Nakonec vypíšeme několik zákazníků z proloženého **trénovacího** segmentu s jejich původním příznakem `Role`, predikovaným čerpáním a reziduem. (Vyčleněné řádky `TEST` úmyslně nejsou skórovány, proto filtrujeme na `Role='TRÉNINK'`, abychom zobrazili plně vyplněné řádky.) Toto je typ řádkového výstupu, který by analytik přiložil ke zprávě o monitorování modelu nebo předal dále do systému nastavování limitů.

In [4]:
PROCEDURA TISK data=scored(obs=8) label;
   ŠTÍTEK CustomerID = "ID zákazníka"
         Role = "Skupina"
         Pred_Utilization = "Predikce čerpání úvěru"
         Resid_Utilization = "Reziduum čerpání úvěru";
   KDE Role = 'TRÉNINK';
   PROMĚNNÁ CustomerID Role Pred_Utilization Resid_Utilization;
SPUSTIT;


No observations in dataset.




NOTE: PROC PRINT data=scored



## Interpretace výsledků

Tabulka **Percent Variation Accounted for** ukazuje, že samotný první faktor pohltí zhruba tři čtvrtiny variability prediktorů (74.16 %, dominantní kolineární směr „stress“), zatímco druhý faktor vysvětluje většinu variability *odezev* (45.29 %, oproti 25.52 % z prvního a jen 2.89 % ze třetího) — typický rys PLS, které přesměrovává komponenty k predikci místo čisté X-variability. Při osmi faktorech se R-kvadráty pro jednotlivé odezvy ustálí na 0.81 (Utilization), 0.78 (DTIChange) a 0.74 (DefaultProp), což potvrzuje, že tři ukazatele úvěrového rizika jsou dobře zachyceny nízkodimenzionální latentní strukturou.

Rozhodujícím faktorem je zde **van der Voetův křížově validovaný test PRESS**: hodnota Root Mean PRESS na vyčleněném segmentu prudce klesá v průběhu prvních čtyř faktorů (6.23 → 4.11 → 3.96 → 3.77) a poté se vyrovná a mírně kolísá nahoru (3.79 → 3.79 → 3.82 → 3.83), takže test vybírá jako úsporný model **čtyři faktory** (nejmenší počet faktorů s p > 0.1 jsou pak dva). Extrakce dalších faktorů by pronásledovala šum v širokém bloku bureau a zhoršila výkon mimo trénovací vzorek — přesně ten typ přeučení, kterému se model úvěrového rizika musí před nasazením vyhnout.

Koeficienty `SOLUTION` poskytují bance interpretovatelnou, regularizovanou lineární bodovací kartu v původní škále proměnných, přičemž `RatePctChg` (≈0.80, 0.88, 0.62 napříč třemi ukazateli) a `InquiryCount` (≈0.47, 0.36, 0.58) se ukazují jako nejsilnější jednotlivé hnací faktory. V praxi by modelář nyní přeproložil model s `nfac=4`, sledoval rezidua v datové sadě `scored` a faktorové skóre nebo koeficienty zavedl do produkčního rozhodovacího pipeline pro riziko.